# पाठ १० - उत्पादनमा AI एजेन्टहरू

यस पाठमा तपाइँ Microsoft Agent Framework (Python) प्रयोग गरेर AI एजेन्टहरूको **उत्पादन पद्धतिहरू** सिक्नु हुनेछ। हामीले समेटेका छौं:

- **अवलोकन क्षमता** — एजेन्ट अन्तरक्रियामा समय निर्धारण र लगिंग थप्ने
- **मूल्यांकन** — प्रतिक्रिया गुणस्तर स्कोर गर्न मूल्यांकनकर्ता एजेन्ट प्रयोग गर्ने
- **लागत व्यवस्थापन** — टोकन अनुकूलन र मोडेल चयनका रणनीतिहरू

परिदृश्य एक **यात्रा एजेन्ट** को हो जसले प्रयोगकर्ताहरूलाई यात्राको योजना बनाउन मद्दत गर्दछ, जसमा निगरानी र मूल्यांकन थपिएको छ।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## उत्पादन विचारहरू

एआई एजेन्टहरूलाई प्रोटोटाइपबाट उत्पादनमा सार्न तीन स्तम्भहरूमा सावधानीपूर्वक ध्यान दिनु आवश्यक छ:

1. **देखिने क्षमता** — तपाईंले एजेन्ट के गर्दैछ, कति समय लिन्छ, र कुन उपकरणहरू कल गर्छ भन्ने देख्न सक्नुपर्छ। ट्रेसिङ र लगिङ बिना उत्पादन समस्याहरू डिबग गर्न लगभग असम्भव हुन्छ।

2. **मूल्याङ्कन** — स्वचालित गुणस्तर जाँचहरूले एजेन्टका प्रतिक्रिया समयका साथ सही, पुरा, र सहायक रहन्छन् भनेर सुनिश्चित गर्छ। एक मूल्याङ्कनकर्ता एजेन्टले परिभाषित मापदण्डहरूसँग प्रतिक्रियाहरू स्कोर गर्न सक्छ।

3. **खर्च व्यवस्थापन** — टोकन प्रयोगले सिधै खर्चमा प्रभाव पार्छ। प्रॉम्प्ट अप्टिमाइजेसन, मोडेल चयन, र क्यासिङ जस्ता रणनीतिहरूले गुणस्तर गुमाउनु बिना खर्चलाई नियन्त्रणमा राख्न मद्दत गर्छन्।


## एक अवलोकनीय एजेन्ट निर्माण गर्दै

हामी यात्रा उपकरणहरू परिभाषित गर्दछौं र एजेन्ट कललाई टाइमिङसँग र्याप गर्दछौं ताकि हामी विलम्बता अनुगमन गर्न सकौँ। उत्पादनमा तपाईंले OpenTelemetry वा समान ट्रेसिङ ब्याकएण्डसँग एकीकृत गर्नुहुनेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्याङ्कन ढाँचाहरू

एक सामान्य उत्पादन ढाँचा दोस्रो एजेन्टलाई **मूल्यांकनकर्ता** को रूपमा प्रयोग गर्ने हो। मूल्यांकनकर्ताले प्राथमिक एजेन्टको प्रतिक्रिया पूर्वनिर्धारित मापदण्डहरू जस्तै पूर्णता, शुद्धता, र उपयोगिताको आधारमा मूल्यांकन गर्दछ।

यसले सक्षम बनाउँछ:
- प्रयोगकर्ताहरूलाई प्रतिक्रिया पुग्नु अघि स्वचालित गुणस्तर ढोका
- प्रॉम्प्टहरू वा मोडेलहरू परिवर्तन हुँदा पुनरावर्तन पत्ता लगाउने
- समयक्रमसँगै एजेन्टको प्रदर्शनको निरन्तर अनुगमन


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## लागत व्यवस्थापन रणनीतिहरू

उत्पादन AI एजेन्टहरूको लागि लागत नियन्त्रण अत्यन्त महत्वपूर्ण छ। यहाँ केन्द्रीय रणनीतिहरू छन्:

| रणनीति | विवरण |
|---|---|
| **प्रॉम्प्ट अनुकूलन** | प्रणाली निर्देशनहरू छोटो राख्नुहोस्। इनपुट टोकनहरू घटाउन दोहोरिएको सन्दर्भ हटाउनुहोस्। |
    "| **मोडेल चयन** | सरल कार्यहरू जस्तै वर्गीकरण वा निष्कर्षणका लागि साना, सस्तो मोडेलहरू (जस्तै GPT-5-mini) प्रयोग गर्नुहोस्, र जटिल तर्कका लागि ठूला मोडेलहरू आरक्षित गर्नुहोस्। |\n",
| **क्याचिङ** | उपकरणको परिणामहरू र बारम्बार सोधिने प्रश्नहरू क्यास गर्नुहोस् जसले दोहोरिएको API कलहरूबाट बचाउँछ। |
| **टोकन बजेटहरू** | अप्रत्याशित लामो प्रतिक्रिया रोक्न `max_tokens` सीमा सेट गर्नुहोस्। |
| **व्याचिङ** | सम्भव भएपछि एकै API कलमा धेरै प्रयोगकर्ता प्रश्नहरू समूहबद्ध गर्नुहोस्। |

व्यवहारमा, स्तरीय दृष्टिकोण राम्ररी काम गर्छ: सरल अनुरोधहरूलाई छिटो र सस्तो मोडेलमा निर्देशित गर्नुहोस् र जटिल प्रश्नहरूलाई मात्र बढी सक्षम मोडेलमा बढाउनुहोस्।


## सारांश

यस पाठमा तपाईंले सिक्नुभयो कि:

1. एजेन्ट अन्तरक्रियामा टाइमिङ र लगिङ सहित **पर्यवेक्षण थप्ने**, ट्रेसिङ र अनुगमनको आधार तयार गर्ने।
2. एउटा मूल्यांकन गर्ने एजेन्टको प्रयोग गरेर स्वचालित रूपमा एजेन्ट प्रतिक्रियाहरूको पूर्णता, शुद्धता, र सहयोगीता **मूल्यांकन गर्ने**।
3. प्रॉम्प्ट अनुकूलन, मोडेल छनोट, क्यासिङ, र टोकन बजेटहरू मार्फत **खर्चहरू व्यवस्थापन गर्ने**।

यी उत्पादन नमूनाहरूले तपाईंका एआई एजेन्टहरूलाई ढुक्क, मापनयोग्य, र व्यापक रूपमा लागत-प्रभावकारी बनाउन मद्दत गर्छन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
